# Studi Kasus 3 — RAG dengan Ragam Chunking

Notebook membandingkan fixed-size, sentence, recursive, structure-aware, dan semantic chunking. Generation sengaja diletakkan setelah retrieval evaluation agar kontribusi chunking tidak tertutup perilaku LLM.


In [ ]:
!pip -q install -U sentence-transformers pandas


In [ ]:
import re, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

document = """# Tokenisasi
Tokenisasi mengubah teks menjadi token. Word tokenization mudah dibaca tetapi memiliki masalah OOV.
# Embedding
Embedding memetakan token atau kalimat menjadi vektor padat. Cosine similarity membandingkan arah vektor.
# RAG
RAG mengambil potongan dokumen relevan sebelum LLM menjawab. Kualitas retrieval dipengaruhi batas chunk dan top-k.
# Evaluasi
Recall@k mengukur apakah konteks relevan ditemukan. MRR mempertimbangkan posisi hasil relevan pertama."""

def fixed_chunks(text,size=100,overlap=20):
    return [text[i:i+size] for i in range(0,len(text),size-overlap)]

def sentence_chunks(text,max_sentences=2):
    s=[x.strip() for x in re.split(r'(?<=[.!?])\s+',text.replace('#','')) if x.strip()]
    return [' '.join(s[i:i+max_sentences]) for i in range(0,len(s),max_sentences)]

def recursive_chunks(text,max_chars=150):
    parts=re.split(r'\n+|(?<=[.!?])\s+',text)
    out,current=[],''
    for part in parts:
        if len(current)+len(part)+1>max_chars and current:
            out.append(current.strip());current=part
        else: current=(current+' '+part).strip()
    if current:out.append(current)
    return out

def structure_chunks(text):
    sections=re.split(r'(?m)^# ',text)
    return [s.strip() for s in sections if s.strip()]

strategies={"fixed":fixed_chunks(document),"sentence":sentence_chunks(document),
            "recursive":recursive_chunks(document),"structure":structure_chunks(document)}
pd.DataFrame([{"strategy":k,"n_chunks":len(v),"avg_chars":np.mean([len(x) for x in v])} for k,v in strategies.items()])


## Semantic chunking

Boundary dibuat ketika similarity kalimat bertetangga turun di bawah threshold. Pada corpus nyata, threshold dikalibrasi dan panjang minimum/maksimum tetap diperlukan.


In [ ]:
model=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
sentences=[x.strip() for x in re.split(r'(?<=[.!?])\s+',document.replace('#','')) if x.strip()]
emb=model.encode(sentences,normalize_embeddings=True)
adjacent=(emb[:-1]*emb[1:]).sum(axis=1)

def semantic_chunks(threshold=0.35):
    out,current=[],[sentences[0]]
    for i,score in enumerate(adjacent):
        if score<threshold: out.append(' '.join(current));current=[]
        current.append(sentences[i+1])
    out.append(' '.join(current));return out
strategies["semantic"]=semantic_chunks()


## Retrieval comparison


In [ ]:
queries=[
 {"q":"Apa metrik untuk memeriksa konteks relevan?","keyword":"Recall@k"},
 {"q":"Mengapa word tokenization bermasalah?","keyword":"OOV"},
 {"q":"Bagaimana RAG menjawab pertanyaan?","keyword":"sebelum LLM"}
]

rows=[]
for name,chunks in strategies.items():
    ce=model.encode(chunks,normalize_embeddings=True)
    for item in queries:
        qe=model.encode(item["q"],normalize_embeddings=True)
        scores=ce@qe
        rank=np.argsort(scores)[::-1]
        relevant=[i for i,c in enumerate(chunks) if item["keyword"].lower() in c.lower()]
        first=next((r+1 for r,i in enumerate(rank) if i in relevant),None)
        rows.append({"strategy":name,"query":item["q"],"hit@2":any(i in relevant for i in rank[:2]),
                     "reciprocal_rank":0 if first is None else 1/first})
result=pd.DataFrame(rows)
result.groupby("strategy")[["hit@2","reciprocal_rank"]].mean().sort_values("reciprocal_rank",ascending=False)


## Analisis

Tambahkan overlap, minimum/maximum length, dokumen jamak, dan ground-truth relevant passage. Laporkan Recall@k, MRR, context precision, redundancy, latency, serta contoh chunk yang kehilangan konteks atau mencampur topik.
